## Popcorn Hack: Casting and Division

Predictions, made before running:

| Question | Prediction | Why |
| --- | --- | --- |
| Q1 `10 / 4` | `2` | `int / int` drops the decimal |
| Q2 `(double) 10 / 4` | `2.5` | the cast happens first, so this is double division |
| Q3 `(int) 7.8` | `7` | casting truncates and doesn't round |
| Q4 `(int) (7.8 + 0.5)` | `8` | adding 0.5 before truncating rounds up |
| Q5 `Integer.MAX_VALUE + 1` | `-2147483648` | int overflow wraps to the minimum |

Hand trace: 10 = 4 * 2 + 2, so integer division gives 2. Promoting 10 to 10.0 preserves the fractional 0.5. Truncating 7.8 removes .8; adding 0.5 first gives 8.3, which truncates to 8. The maximum 32-bit integer is 2147483647; adding one wraps to -2147483648. The additional underflow prediction is that MIN_VALUE - 1 equals MAX_VALUE.


In [1]:
public class CodeRunner1_PopcornHack {
    public static void main(String[] args) {
        // Q1: 10 / 4 = 2 remainder 2; integer division gives 2.
        System.out.println("Q1: " + (10 / 4));
        // Q2: promote 10 to 10.0, then divide: 2.5.
        System.out.println("Q2: " + ((double) 10 / 4));
        // Q3: truncate 7.8 toward zero: 7.
        System.out.println("Q3: " + (int) 7.8);
        // Q4: 7.8 + 0.5 = 8.3; truncate to 8.
        System.out.println("Q4: " + (int) (7.8 + 0.5));
        // Q5: exceeding the 32-bit maximum wraps to the minimum.
        int x = Integer.MAX_VALUE;
        System.out.println("Q5: " + (x + 1));
        System.out.println("Popcorn checks: " + (10 / 4 == 2 && (double) 10 / 4 == 2.5
            && (int) 7.8 == 7 && (int) (7.8 + 0.5) == 8 && x + 1 == Integer.MIN_VALUE));
        // Additional boundary prediction: minimum minus one wraps to maximum.
        int min = Integer.MIN_VALUE;
        System.out.println("Underflow check: " + (min - 1 == Integer.MAX_VALUE));
    }
}

CodeRunner1_PopcornHack.main(null);


Q1: 2
Q2: 2.5
Q3: 7
Q4: 8
Q5: -2147483648
Popcorn checks: true
Underflow check: true


## Homework Hack: Casting Methods

Predictions before execution: averages (5,10), (3,4), and (-3,2) give 7.5, 3.5, and -0.5. Equal MAX or MIN operands average to that boundary without wrapping. Percentages 80/100 and 7/10 give 80.0 and 70.0; a zero total gives 0.0, and MAX/MAX gives 100.0. Rounding 5.4, 5.5, -5.4, -5.5, 0, and -0.5 gives 5, 6, -5, -6, 0, and -1. Exact int boundaries remain unchanged.

`averageAsDouble` promotes a before addition so even a+b at the integer limits cannot overflow. Dividing the double sum by 2.0 preserves the fractional part. `percentScore` first returns 0.0 for total zero; otherwise multiplying by 100.0 makes the arithmetic floating-point before division. `nearestInteger` checks the sign and shifts by +0.5 or -0.5 before truncating toward zero; ties therefore round away from zero. The check helper compares every actual result with its expected value and throws an error on a failure.

The output below was produced by running these Java class bodies with local Java 17. The notebook uses the Java-kernel main invocation. Site runner network failures are recorded separately and are not represented as successful site runs.


In [2]:
public class CodeRunner2_CastingPractice {
    public static double averageAsDouble(int a, int b) {
        // Promote BEFORE addition, avoiding int overflow and preserving fractions.
        return ((double) a + b) / 2.0;
    }
    public static double percentScore(int correct, int total) {
        // An empty total returns 0.0, as specified, instead of dividing by zero.
        if (total == 0) return 0.0;
        // A double multiplier prevents integer division and int multiplication overflow.
        return (100.0 * correct) / total;
    }
    public static int nearestInteger(double value) {
        // Truncation moves toward zero; shift half a unit away from zero first.
        if (value >= 0) return (int) (value + 0.5);
        return (int) (value - 0.5);
    }
    private static void check(String label, double actual, double expected) {
        boolean pass = actual == expected;
        System.out.println(label + " = " + actual + " | " + (pass ? "PASS" : "FAIL"));
        if (!pass) throw new AssertionError(label);
    }
    public static void main(String[] args) {
        check("Average of 5 and 10", averageAsDouble(5, 10), 7.5);
        check("Average of 3 and 4", averageAsDouble(3, 4), 3.5);
        check("Average of -3 and 2", averageAsDouble(-3, 2), -0.5);
        check("Average at MAX boundary", averageAsDouble(Integer.MAX_VALUE, Integer.MAX_VALUE), 2147483647.0);
        check("Average at MIN boundary", averageAsDouble(Integer.MIN_VALUE, Integer.MIN_VALUE), -2147483648.0);
        check("80 out of 100 percent", percentScore(80, 100), 80.0);
        check("7 out of 10 percent", percentScore(7, 10), 70.0);
        check("0 out of 0 percent", percentScore(0, 0), 0.0);
        check("5 out of 0 percent", percentScore(5, 0), 0.0);
        check("MAX out of MAX percent", percentScore(Integer.MAX_VALUE, Integer.MAX_VALUE), 100.0);
        check("Round 5.4", nearestInteger(5.4), 5);
        check("Round 5.5", nearestInteger(5.5), 6);
        check("Round -5.4", nearestInteger(-5.4), -5);
        check("Round -5.5", nearestInteger(-5.5), -6);
        check("Round zero", nearestInteger(0), 0);
        check("Round -0.5", nearestInteger(-0.5), -1);
        check("Round MAX boundary", nearestInteger(Integer.MAX_VALUE), Integer.MAX_VALUE);
        check("Round MIN boundary", nearestInteger(Integer.MIN_VALUE), Integer.MIN_VALUE);
    }
}

CodeRunner2_CastingPractice.main(null);


Average of 5 and 10 = 7.5 | PASS
Average of 3 and 4 = 3.5 | PASS
Average of -3 and 2 = -0.5 | PASS
Average at MAX boundary = 2.147483647E9 | PASS
Average at MIN boundary = -2.147483648E9 | PASS
80 out of 100 percent = 80.0 | PASS
7 out of 10 percent = 70.0 | PASS
0 out of 0 percent = 0.0 | PASS
5 out of 0 percent = 0.0 | PASS
MAX out of MAX percent = 100.0 | PASS
Round 5.4 = 5.0 | PASS
Round 5.5 = 6.0 | PASS
Round -5.4 = -5.0 | PASS
Round -5.5 = -6.0 | PASS
Round zero = 0.0 | PASS
Round -0.5 = -1.0 | PASS
Round MAX boundary = 2.147483647E9 | PASS
Round MIN boundary = -2.147483648E9 | PASS
